Imports

In [4]:
#Log in to huggingface hub
from huggingface_hub import login
import os
from dotenv import load_dotenv
load_dotenv()
login(token = os.getenv("HFToken"))
#HFToken = [your token] in .env

In [50]:
""" Official evaluation script for v1.1 of the SQuAD dataset. """

import re
import string
from collections import Counter


def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""

    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def f1_score(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

In [5]:
#Model Initiallization
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id,padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:0",dtype=torch.bfloat16)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [47]:
#Dataset Initiallization
from datasets import load_dataset
ds = load_dataset("rajpurkar/squad")

In [41]:
#Prompt Generation
prompt = [[
    {
        "role": "system",
        "content":"Answer the problem in one sentence"
        #"content": "You are a precise calculator. Output ONLY the final numerical answer. Do not include words, units, explanations, or punctuation."
    },
    {
        "role": "user",
        "content":  f"Context:\n{context}\n\nQuestion:\n{question}"
    }
]for context,question in zip(ds["train"]["context"],ds["train"]["question"])]
questions = 50
input = tokenizer.apply_chat_template(
    prompt[0:questions],
    tokenize = True,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt",
    clean_up_tokenization_spaces=False
).to("cuda:0")
input_length = input["input_ids"].shape[1]

In [42]:
#Output Saving
import json
from transformers import logging
logging.set_verbosity_info()
raw_output = model.generate(
    **input,
    max_new_tokens= 100,
    do_sample=False,
    temperature=1,
    pad_token_id=tokenizer.pad_token_id,
    tokenizer=tokenizer,
    return_dict_in_generate=True,
    output_logits=True,
    output_scores=True
    )
output = raw_output.sequences
#for i in range(questions):
 #   write.append(tokenizer.decode(output[i][input_length:],skip_special_tokens=True))
#json.dump(write,f)

In [43]:
#Answer Search
write = []
for i in range(questions):
   write.append(tokenizer.decode(output[i][input_length:],skip_special_tokens=True,clean_up_tokenization_spaces=True))
with open("result.json","w") as f:
    json.dump(write,f)

In [59]:
with open("result.json","r") as f:
    data = json.load(f)
for i in range(questions):
    print(f1_score(data[i],ds["train"]["answers"][i]["text"][0]))

In [60]:
logits = raw_output.scores
probability = torch.softmax(logits[0],dim=0)
id = torch.argmax(probability[0])
max_probability = probability[0][id]


In [62]:
max_probability

tensor(0.9886, device='cuda:0')